# <font color="green"> **AI for Cybersecurity Project 2024-25 Group 9**

<font color="black">
Lamb Giovanni - 0622702168 <br>
Orlando Palma - 0622702433 <br>
Saturnino Fabrizio - 0622702328 <br>
Zottarelli Egidio - 0622702175

## **Contents**
1. Transferability <br>
    1.1 FGSM <br>
    1.2 BIM <br>
    1.3 PGD <br>
    1.4 DeepFool <br>
    1.5 Carlini & Wagner <br>



## **Loading prereqs**

In [ ]:
# Local
!pip install git+https://github.com/rcmalli/keras-vggface.git
!pip install tensorflow==2.12 keras==2.12 keras_applications
!pip install facenet-pytorch
!pip install adversarial-robustness-toolbox[all]

In [ ]:
# Kaggle-compatible install + patch for keras_vggface
!pip install git+https://github.com/rcmalli/keras-vggface.git
!pip install numpy==1.23.5 tensorflow==2.12 keras==2.12 keras_applications mtcnn

# Patch keras_vggface to work with Keras 2.12
!python3 -c "import keras_vggface, os; file=os.path.join(keras_vggface.__path__[0], 'models.py'); content=open(file).read().replace('from keras.engine.topology import get_source_inputs', 'from keras.utils.layer_utils import get_source_inputs'); open(file, 'w').write(content)"

!pip install facenet-pytorch
!pip install adversarial-robustness-toolbox[all]
!pip install pillow==9.5.0 --force-reinstall

## **Import**

In [ ]:
import os
import torch
import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import art
import torch
import logging

from datetime import datetime
from PIL import Image
from tqdm import tqdm
from facenet_pytorch import InceptionResnetV1
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import (
    ProjectedGradientDescent,
    BasicIterativeMethod,
    FastGradientMethod,
    DeepFool,
    CarliniL2Method,
    CarliniLInfMethod
)
from keras_vggface.vggface import VGGFace
from keras_vggface.utils import preprocess_input
from keras_vggface.utils import decode_predictions

C:\Users\orlan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## **1. Transferability**
The goal of this phase is to investigate the **transferability** of adversarial examples generated for classifier NN1 to classifier NN2. 

> The concept of transferability in adversarial attacks refers to the ability of adversarial examples generated for one model to deceive another model. By testing the same set of adversarial examples against both NN1 and NN2, we aim to evaluate whether the vulnerabilities exploited by adversarial attacks generalize across different classifiers. 

This investigation is crucial for assessing the robustness and generalization capability of NN2 against adversarial attacks. If adversarial examples crafted for NN1 consistently lead to misclassifications in NN2, it indicates a shared vulnerability or similarity in decision boundaries between the two classifiers. Conversely, if NN2 demonstrates resilience against the same adversarial examples, it suggests differences in how NN2 processes inputs compared to NN1, potentially highlighting NN2's robustness against adversarial perturbations. To investigate the transferability, we analyze and compare the **Security Evaluation Curves** (SECs) of NN1 and NN2 using adversarial images generated for NN1 through various attack methods: FGSM (Fast Gradient Sign Method), BIM (Basic Iterative Method), PGD (Projected Gradient Descent), DeepFool, and Carlini-Wagner attacks. These SECs provide a graphical depiction of each network's resilience to adversarial attacks, showing how their classification accuracy changes as the attack strength increases, typically indicated by parameters like epsilon. This comparison seeks to investigate whether adversarial examples crafted to deceive NN1 also impact NN2 similarly across different attack types, thereby assessing the transferability of adversarial vulnerabilities between the two classifiers.

In [ ]:
class ExperimentManager:
    def __init__(self, attacker, param_name, param_values, victim=None, preprocess=None, save_path=".", save_adv_samples=True):
        """
        Initialize the ExperimentManager.

        Args:
            attacker: The adversarial attack object (e.g., DeepFool, FGSM).
            param_name: The name of the attack parameter to vary (e.g., 'epsilon').
            param_values: List of values for the attack parameter.
            victim: (Optional) The victim model to attack. If None, uses attacker's estimator.
            preprocess: (Optional) Preprocessing function to apply to inputs before prediction.
            save_path: (Optional) Directory to save experiment results.
            save_adv_samples: (Optional) Whether to save adversarial samples generated during the run.
        """
        self._attacker = attacker
        self._victim = victim if victim is not None else attacker.estimator
        self.param_name = param_name
        self.param_values = param_values
        self.save_path = save_path
        self._data = None
        self.save_adv_samples = save_adv_samples
        self._adv_samples = None
        self.preprocess = preprocess if victim is not None else None
        if victim is not None and preprocess is None:
            raise ValueError("You must specify a preprocessing function if you pass a victim model.")

    @property
    def experiment_name(self):
        """
        Returns a unique experiment name based on the attack, parameter, and current timestamp.
        """
        attack_name = self._attacker.__class__.__name__.lower()
        return f"{attack_name}_{self.param_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    @property
    def adversarial_samples(self):
        """
        Returns the adversarial samples generated during the experiment.
        """
        return self._adv_samples

    @property
    def security_evaluation_data(self):
        """
        Returns the data collected during the security evaluation (e.g., accuracy, perturbation).
        """
        return self._data

    def set_param_name(self, param_name):
        """
        Set the name of the attack parameter to vary.

        Args:
            param_name: The new parameter name.
        """
        self.param_name = param_name

    def set_param_values(self, param_values):
        """
        Set the list of values for the attack parameter.

        Args:
            param_values: List of new parameter values.
        """
        self.param_values = param_values

    def save(self, experiment_name=None):
        """
        Save the experiment data to a pickle file.

        Args:
            experiment_name: (Optional) Name for the saved experiment file.
        """
        if experiment_name is None:
            experiment_name = self.experiment_name
        os.makedirs(self.save_path, exist_ok=True)
        with open(os.path.join(self.save_path, f"{experiment_name}.pkl"), 'wb') as file:
            pickle.dump(self._data, file)

    def load_adversarial_samples(self, experiment_folder, prefix="adv"):
        """
        Load adversarial samples saved as .npy files for each value of param_name.

        Args:
            experiment_folder (str): Path to the folder where .npy files are saved.
            prefix (str): Prefix for the file names (default: 'adv').
        """
        self._adv_samples = []
        for param_value in self.param_values:
            filename = f"{prefix}_{self.param_name}{param_value}.npy"
            filepath = os.path.join(experiment_folder, filename)
            if not os.path.exists(filepath):
                raise FileNotFoundError(f"File non trovato: {filepath}")
            adv_data = np.load(filepath)
            self._adv_samples.append(adv_data)


    def run(self, x_test, y_test, targeted=False, target_class=None, num_classes=None, norm='inf', save_run=True, experiment_name=None):
        """
        Run the experiment by generating adversarial samples for each parameter value and evaluating the model.

        Args:
            x_test: Test input samples (numpy array).
            y_test: True labels for the test samples.
            targeted: (Optional) Whether to perform a targeted attack (default: False).
            target_class: (Optional) Target class for targeted attacks.
            num_classes: (Optional) Number of classes (required for targeted attacks).
            norm: (Optional) Norm type for perturbation calculation ('inf', 1, or 2).
            save_run: (Optional) Whether to save the experiment results (default: True).
            experiment_name: (Optional) Name for saving the experiment results.
        """
        self._data = {
            'param_values': self.param_values,
            'norm': None,
            'nb_pred': len(x_test),
        }

        if experiment_name is None:
            experiment_name = self.experiment_name

        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
        logger = logging.getLogger(__name__)
        
        n_runs = len(self.param_values)
        nb_correct = []
        perturbation = [None] * n_runs
        self._adv_samples = [None] * n_runs if self.save_adv_samples else None

        if isinstance(self._attacker, DeepFool):
            perturbation_mean = [None] * n_runs
            for run, param_value in enumerate(self.param_values):
                print(f"\n Running DeepFool with {self.param_name}={param_value}")
                self._attacker.set_params(**{self.param_name: param_value})
                adv_samples = []
                correct = 0
                pert_list = []

                for i in range(len(x_test)):
                    x_i = x_test[i:i+1]
                    # Preprocess input sample (your custom preprocessing)
                    x_input = self.preprocess(x_i) if self.preprocess is not None else x_i

                    # If the preprocess returns (N, H, W, C), but DeepFool expects (N, C, H, W)
                    if isinstance(x_input, np.ndarray) and x_input.ndim == 4 and x_input.shape[-1] == 3:
                        x_input = np.transpose(x_input, (0, 3, 1, 2))

                    # Copy to avoid issues with negative strides
                    x_input = x_input.copy()

                    # Generate adversarial example
                    x_adv = self._attacker.generate(x_input)

                    # Save adversarial sample in original format (N, C, H, W)
                    adv_samples.append(x_adv[0])

                    if self.preprocess is not None and self._victim != self._attacker.estimator:
                        # x_adv is (N, C, H, W) → convert to (N, H, W, C) before preprocessing
                        x_adv_img = np.transpose(x_adv, (0, 2, 3, 1))
                        x_adv_input = self.preprocess(x_adv_img)
                    else:
                        x_adv_input = x_adv

                    # Convert to PyTorch tensor with shape (N, C, H, W)
                    x_adv_tensor = torch.from_numpy(np.transpose(x_adv_input[0].copy(), (2, 0, 1))).unsqueeze(0).to(device)
                    
                    with torch.no_grad():
                        x_adv_np = x_adv_tensor.cpu().numpy()  # (N, 3, 224, 224)
                        x_adv_np = np.transpose(x_adv_np, (0, 2, 3, 1))  # now (N, 224, 224, 3)
                        output = self._victim.predict(x_adv_np)
                    pred = np.argmax(output, axis=1)[0]

                    y_i = y_test[i]
                    if pred == y_i:
                        correct += 1

                    # Compute max L∞ perturbation
                    # Note: x_adv and x_i are both (C,H,W), ensure they have the same shape
                    pert_list.append(np.max(np.abs(x_adv[0] - x_input[0])))

            
                adv_samples = np.array(adv_samples)
                pert_list = np.array(pert_list)

                if self.save_adv_samples:
                    self._adv_samples[run] = adv_samples

                nb_correct.append(correct)
                perturbation[run] = np.max(pert_list)
                perturbation_mean[run] = np.mean(pert_list)

                # Extra diagnostics
                n_valid = np.sum(pert_list < 0.1)
                print(f"→ {n_valid}/{len(pert_list)} images with L∞ < 0.1")
                print(f"   {self.param_name} = {param_value}")
                print(f"   Mean perturbation = {perturbation_mean[run]:.4f}, max = {pert_list.max():.4f}")

            self._data['nb_correct'] = nb_correct
            self._data['perturbation'] = perturbation
            self._data['perturbation_mean'] = perturbation_mean

        else:
            nb_correct_targeted = [None] * n_runs if targeted else None
            for i, param_value in tqdm(enumerate(self.param_values), total=n_runs):
                self._attacker.set_params(**{self.param_name: param_value})

                if targeted:
                    # For targeted attacks, create one-hot encoded target labels
                    targeted_labels = target_class * np.ones(len(y_test))
                    one_hot_targeted_labels = tf.keras.utils.to_categorical(targeted_labels, num_classes=num_classes)
                    x_test_adv = self._attacker.generate(np.array(x_test), one_hot_targeted_labels)
                else:
                    x_test_adv = self._attacker.generate(np.array(x_test))

                if self.save_adv_samples:
                    self._adv_samples[i] = x_test_adv
                    if norm == 1:
                        perturbation[i] = np.mean(np.abs(x_test_adv - x_test))
                    elif norm == 2:
                        perturbation[i] = np.linalg.norm(x_test_adv - x_test)
                    elif norm == 'inf':
                        perturbation[i] = np.max(np.abs(x_test_adv - x_test))

                x_test_adv_pred = np.argmax(self._victim.predict(
                    self.preprocess(x_test_adv) if self._victim != self._attacker.estimator else x_test_adv), axis=1)

                nb_correct[i] = np.sum(x_test_adv_pred == np.array(y_test))

                if targeted:
                    nb_correct_targeted[i] = np.sum(x_test_adv_pred == target_class)

            self._data['perturbation'] = perturbation
            self._data['nb_correct'] = nb_correct
            if targeted:
                self._data['nb_correct_targeted'] = nb_correct_targeted

        if save_run:
            self.save(experiment_name)

    def plot(self, save=False):
        """
        Plot the accuracy and perturbation curves for the experiment.

        Args:
            save: (Optional) Whether to save the plots as image files.
        """
        plt.style.use("seaborn-v0_8-whitegrid")
        fig, ax = plt.subplots()
        y_values = 100 * (np.array(self._data['nb_correct']) / self._data['nb_pred'])
        ax.plot(np.array(self.param_values), y_values, 'r--', label='Classifier')

        if 'nb_correct_targeted' in self._data and self._data['nb_correct_targeted'] is not None:
            y_values_targeted = 100 * (np.array(self._data['nb_correct_targeted']) / self._data['nb_pred'])
            ax.plot(np.array(self.param_values), y_values_targeted, 'b--', label='Target Class')

        plt.xlabel(self.param_name)
        plt.ylabel("Accuracy (%)")
        ax.legend(loc='upper right', fontsize='large', frameon=True, edgecolor='black')
        ax.grid(True)
        plt.title('Security Evaluation Curve')
        plt.show()

        if save:
            plt.savefig(os.path.join(self.save_path, f"{self.experiment_name}.png"))

        # Perturbation plot
        if 'perturbation' in self._data and self._data['perturbation'] is not None:
            fig2, ax2 = plt.subplots()
            ax2.plot(np.array(self.param_values), np.array(self._data['perturbation']), 'g--', label='L∞ Perturbation')
            ax2.set_xlabel(self.param_name)
            ax2.set_ylabel("L∞ Perturbation Magnitude")
            ax2.set_title("Perturbation Curve")
            ax2.legend(loc='upper left', fontsize='large', frameon=True, edgecolor='black')
            ax2.grid(True)
            plt.show()

            if save:
                plt.savefig(os.path.join(self.save_path, f"{self.experiment_name}_perturbation.png"))

In [4]:
# Initialize the Inception Resnet V1 model for face recognition
nn1 = InceptionResnetV1(pretrained='vggface2').eval().to(device)
nn1.classify = True                                                          # Enable classification mode (return logits)

In [ ]:
# Initialize the classifier with the Inception Resnet V1 model
target_class = 1050
num_classes = 8631

classifier = PyTorchClassifier(
    model=nn1,                                                     # The PyTorch model to use
    clip_values=(-1, 1),                                           # The minimum and maximum values of the input                        
    loss=torch.nn.CrossEntropyLoss(),                              # The loss function
    optimizer=torch.optim.Adam(nn1.parameters(), lr=0.001),        # The optimizer
    input_shape=(3, 160, 160),                                     # The shape of the input
    nb_classes=num_classes,                                        # The number of classes
)

In [6]:
nn2 = VGGFace(model='resnet50')

In [ ]:
def preprocess_face(face, required_size=(224, 224)):
    """
    Preprocess a single face image for VGGFace/ResNet50.

    Args:
        face: Input image, shape can be (3, H, W) or (H, W, 3), values in [-1, 1] or uint8.
        required_size: Tuple (height, width) to resize the image to (default: (224, 224)).

    Returns:
        Preprocessed face image as a float32 numpy array of shape (224, 224, 3).
    """
    # If color channel is first (C, H, W), transpose to (H, W, C)
    if face.shape[0] == 3 and face.ndim == 3:
        face = face.transpose(1, 2, 0)  # (H, W, C)

    # Now we expect (H, W, 3)
    if face.shape[-1] != 3:
        raise ValueError(f"Unexpected face shape: {face.shape}, expected (H, W, 3)")

    # Convert to uint8 if needed (assuming input was [-1, 1] float)
    if face.dtype != np.uint8:
        face = ((face + 1.0) * 127.5).astype('uint8')

    # Resize to required size
    face_image = Image.fromarray(face)
    face_image = face_image.resize(required_size)
    face_array = np.asarray(face_image).astype('float32')

    return face_array


def preprocess(images):
    """
    Preprocess a batch of images for VGGFace/ResNet50.

    Args:
        images: List or array of images, each of shape (3, H, W) or (H, W, 3), values in [-1, 1] or uint8.

    Returns:
        Preprocessed images as a numpy array of shape (N, 224, 224, 3).
    """
    # Preprocess each image in the batch
    images = [preprocess_face(img) for img in images]

    # Stack into a single array: shape (N, 224, 224, 3)
    images = np.array(images)

    # Apply VGGFace preprocessing (ResNet50 = version 2)
    return preprocess_input(images, version=2)

In [8]:
x_test = np.load('datasets/x_test_32.npy')
y_test = np.load('datasets/y_test_32.npy')

In [ ]:
def evaluate_loaded_samples(manager, y_true):
    """
    Evaluate the victim model (nn2) on loaded adversarial examples.

    Args:
        manager (ExperimentManager): The ExperimentManager object that has loaded adversarial examples.
            - manager.adversarial_samples: List of adversarial sample arrays, one for each parameter value.
            - manager.param_name: Name of the attack parameter varied (e.g., 'eps').
            - manager.param_values: List of parameter values corresponding to each adversarial set.
            - manager.preprocess: Preprocessing function to apply to adversarial samples before prediction.
            - manager._victim: The victim model to evaluate (e.g., nn2).
        y_true (np.ndarray): True labels for the test set (length N).

    Returns:
        all_accuracies (list): List of accuracies for each adversarial set.
    """
    all_accuracies = []

    for i, adv in enumerate(manager.adversarial_samples):
        print(f"\nEvaluating on adversarial set with {manager.param_name}={manager.param_values[i]}")
        
        # ART often returns (N, C, H, W) → convert to (N, H, W, C) for Keras
        if adv.shape[1] == 3:
            adv = np.transpose(adv, (0, 2, 3, 1))  # (N, H, W, C)

        # Apply preprocessing (including resize and normalization)
        adv_preprocessed = manager.preprocess(adv)

        # Predict with the victim model (Keras)
        preds = manager._victim.predict(adv_preprocessed, verbose=0)
        y_pred = np.argmax(preds, axis=1)

        acc = np.mean(y_pred == y_true)
        print(f"Accuracy on adversarial examples: {acc:.4f}")
        all_accuracies.append(acc)

    return all_accuracies

### **Fast Gradient Sign Method (FGSM)**
The objective is to evaluate the transferability of adversarial examples generated using the **FGSM** (Fast Gradient Sign Method) attack on NN1 to the performance of NN2. By varying the epsilon parameter in the FGSM attack, we generate adversarial examples and assess NN2's classification accuracy. This analysis aims to determine if adversarial examples crafted for NN1 using FGSM also affect NN2 similarly, thereby exploring the transferability of adversarial vulnerabilities between the two classifiers under different attack intensities.

#### FGSM Untargeted

In [ ]:
eps_range = np.linspace(0, 0.1, 10)

In [ ]:
attack = FastGradientMethod(estimator=classifier, targeted=False)

sec_eval = ExperimentManager(
    attack, 
    param_name="eps", 
    param_values=eps_range, 
    victim=nn2, 
    preprocess=preprocess, 
    save_adv_samples=False, 
    save_path='attacks/FGSM/FGSM_untargeted/nn2')

sec_eval.run(
    x_test, 
    y_test, 
    save_run=True, 
    experiment_name='nn2_fgsm_untargeted_eps_0_to_0.1')

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of FGSM Untargeted attack to ResNet50')
plt.grid(True)
plt.show()

#### FGSM Targeted

In [ ]:
eps_range = np.linspace(0, 0.1, 10)

In [ ]:
attack = FastGradientMethod(estimator=classifier, targeted=True)

sec_eval = ExperimentManager(
    attack, 
    param_name="eps", 
    param_values=eps_range, 
    victim=nn2, 
    preprocess=preprocess, 
    save_adv_samples=True, 
    save_path='attacks/FGSM/FGSM_targeted/nn2')

sec_eval.run(
    x_test, 
    y_test, 
    targeted = True, 
    target_class = target_class, 
    num_classes = num_classes, 
    save_run=True, 
    experiment_name='nn2_fgsm_targeted_eps_0_to_0.1')

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of FGSM Targeted attack to ResNet50')
plt.grid(True)
plt.show()

### **Basic Iterative Method (BIM)**
The objective is to evaluate the transferability of adversarial examples generated using the **BIM** (Basic Iterative Method) attack on NN1 to the performance of NN2. By varying the epsilon parameter in the BIM attack, we generate adversarial examples and assess NN2's classification accuracy. This analysis aims to determine if adversarial examples crafted for NN1 using BIM also affect NN2 similarly, thereby exploring the transferability of adversarial vulnerabilities between the two classifiers under different attack intensities.

#### BIM Untargeted

In [ ]:
eps_step = 0.001
max_iter = 100
eps_range = np.linspace(0, 0.1, 10)

In [ ]:
attack = BasicIterativeMethod(
    estimator=classifier,
    eps_step=eps_step,
    max_iter=max_iter,
    targeted=False
)

sec_eval = ExperimentManager(
    attack,
    param_name = "eps",
    victim=nn2,
    preprocess=preprocess,
    save_adv_samples=True,
    param_values=eps_range, 
    save_path='attacks/BIM/BIM_untargeted/nn2'
)

sec_eval.run(
    x_test, 
    y_test, 
    save_run=True,
    experiment_name='nn2_bim_untargeted_eps_0_to_0.1_epsstep_0.001_maxiter_100')

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of BIM Untargeted attack to ResNet50')
plt.grid(True)
plt.show()

#### BIM Targeted

In [ ]:
eps_step = 0.015
max_iter = 30
eps_range = np.linspace(0, 0.1, 5)

In [ ]:
attack = BasicIterativeMethod(
    estimator=classifier,
    eps_step=eps_step,
    max_iter=max_iter,
    targeted=True
)

sec_eval = ExperimentManager(
    attack,
    param_name = "eps",
    victim=nn2,
    preprocess=preprocess,
    save_adv_samples=True,
    param_values=eps_range,
    save_path='attacks/BIM/BIM_targeted/nn2'
)

sec_eval.run(
    x_test, 
    y_test, 
    save_run=True,
    targeted = True,
    target_class = target_class,
    num_classes = num_classes,
    experiment_name='nn2_bim_targeted_eps_0_to_0.1_epsstep_0.001_maxiter_100')


In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of BIM Targeted attack to ResNet50')
plt.grid(True)
plt.show()

### **Projected Gradient Descent (PGD)**
The objective is to evaluate the transferability of adversarial examples generated using the **PGD** (Projected Gradient Descent) attack on NN1 to the performance of NN2. By varying the epsilon parameter in the PGD attack, we generate adversarial examples and assess NN2's classification accuracy. This analysis aims to determine if adversarial examples crafted for NN1 using PGD also affect NN2 similarly, thereby exploring the transferability of adversarial vulnerabilities between the two classifiers under different attack intensities.

#### PGD Untargeted

In [ ]:
eps_step = 0.05
max_iter = 20
num_random_init = 5
eps_range = np.linspace(0, 0.1, 10)

In [ ]:
attack = ProjectedGradientDescent(
    estimator=classifier,
    eps_step=eps_step,
    max_iter=max_iter,
    num_random_init=num_random_init,
    targeted=False
  )
sec_eval = ExperimentManager(
    attack,
    param_name="eps",
    param_values=eps_range,
    victim=nn2,
    preprocess=preprocess,
    save_adv_samples=True,
    save_path='attacks/PGD/PGD_untargeted/nn2')

sec_eval.run(
    x_test, 
    y_test, 
    save_run=True, 
    experiment_name='nn2_pgd_untargeted_eps_0_to_0.1_steps10_epsstep0.05_maxiter_20_randominit_5')

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of PGD Untargeted attack to ResNet50')
plt.grid(True)
plt.show()

#### PGD Targeted

In [ ]:
eps_step = 0.01
max_iter = 10
num_random_init = 0
eps_range = np.linspace(0, 0.1, 10)

In [ ]:
attack = ProjectedGradientDescent(
    estimator=classifier,
    eps_step=eps_step,
    max_iter=max_iter,
    num_random_init=num_random_init,
    targeted=True
  )

sec_eval = ExperimentManager(
    attack,
    param_name="eps",
    param_values=eps_range,
    victim=nn2,
    preprocess=preprocess,
    save_adv_samples=False,
    save_path='attacks/PGD/PGD_targeted/nn2')

sec_eval.run(
    x_test,
    y_test,
    targeted = True,
    target_class = target_class,
    num_classes = num_classes,
    experiment_name= 'nn2_pgd_targeted_eps_0_to_0.1_steps10_epsstep0.01_maxiter_10_randominit_0')

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of PGD Targeted attack to ResNet50')
plt.grid(True)
plt.show()

### **DeepFool**
The objective is to evaluate the transferability of adversarial examples generated using the **DeepFool** attack on NN1 to the performance of NN2. By varying the max_iter parameter in the DeepFool attack, we generate adversarial examples and assess NN2's classification accuracy. This analysis aims to determine if adversarial examples crafted for NN1 using DeepFool also affect NN2 similarly, thereby exploring the transferability of adversarial vulnerabilities between the two classifiers under different attack intensities.

In [9]:
epsilon = 1e-06
max_iter = 1
nb_grads = 12
batch_size = 20
max_iter_range = [1, 2, 3]

In [ ]:
attack = DeepFool(
    classifier=classifier, 
    epsilon = epsilon,
    max_iter=max_iter,
    nb_grads=nb_grads,
    batch_size = batch_size
)

sec_eval = ExperimentManager(
    attack, 
    param_name="max_iter", 
    param_values=max_iter_range,
    victim=nn2,
    preprocess=preprocess,
    save_adv_samples=True,
    save_path='attacks/DeepFool/nn2'
)

sec_eval.load_adversarial_samples(
    experiment_folder='attacks/DeepFool/nn2/adv_max_iter', 
    prefix="adv")

accuracies = evaluate_loaded_samples(
    sec_eval, 
    y_test, 
    x_test,
    experiment_name="nn2_deepfool_nbgrads12_eps1e-06_maxiter_1_to_3",
    save_results=True
)

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of DeepFool attack to ResNet50')
plt.grid(True)
plt.show()

### **Carlini & Wagner**
The objective is to evaluate the transferability of adversarial examples generated using the **Carlini & Wagner** attack on NN1 to the performance of NN2. By varying the binary search steps in the Carlini & Wagner attack, we generate adversarial examples and assess NN2's classification accuracy. This analysis aims to determine if adversarial examples crafted for NN1 using Carlini & Wagner also affect NN2 similarly, thereby exploring the transferability of adversarial vulnerabilities between the two classifiers under different attack intensities.

#### Carlini Untargeted

In [ ]:
learning_rate = [0.1, 0.04, 0.035, 0.025, 0.05]
initial_const = 0.01
confidence = 0.1
max_iter_range = [1, 2, 3, 5, 7]
decrese_factor = [0.9, 0.8, 0.8, 0.8, 0.8]
largest_const = [5.0, 8.0, 8.0, 8.0, 8.0]
const_factor = [1.2, 2.0, 2.0, 2.0, 2.0]
batch_size = [8, 32, 32, 32, 64]

In [ ]:
max_element= []
for max_iter, learning_rate, decrese_factor, largest_const, const_factor, batch_size in zip(max_iter_range, learning_rate, decrese_factor, largest_const, const_factor, batch_size):
    max_element.append(max_iter)
    attack = CarliniLInfMethod(
      classifier=classifier,
      confidence=confidence,
      learning_rate=learning_rate,
      initial_const=initial_const,
      decrease_factor=decrese_factor,
      largest_const=largest_const, 
      const_factor=const_factor, 
      batch_size = batch_size,
      targeted=False)
    
    sec_eval = ExperimentManager(
        attack, 
        "max_iter", 
        max_element, 
        victim=nn2,
        preprocess=preprocess,
        save_adv_samples=True,
        save_path='attacks/CW/CW_untargeted/nn2')
    
    sec_eval.run(
        x_test, 
        y_test, 
        save_run = True,
        experiment_name='nn2_carlini_untargeted_learningrate_0.05_initial_const_0.5_confidence_0.1_max_iter_1_to_7'
        )
    sec_eval.plot()
    max_element.clear()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of DeepFool attack to ResNet50')
plt.grid(True)
plt.show()

#### Carlini Targeted

In [ ]:
binary_search_steps = 1
confidence = 0.1
learning_rate = 0.05
initial_const = 0.5
max_iter_range = [1,2,3,5,7]

In [ ]:
attack = CarliniL2Method(
    classifier=classifier,
    confidence=confidence,
    binary_search_steps=binary_search_steps,
    learning_rate=learning_rate,
    initial_const=initial_const,
    targeted=True)

sec_eval = ExperimentManager(
    attack, 
    "max_iter", 
    max_iter_range,
    victim=nn2, 
    preprocess=preprocess, 
    save_adv_samples=False, 
    save_path='attacks/CW/CW_targeted/nn2')

sec_eval.run(
    x_test, 
    y_test, 
    targeted = True, 
    target_class = target_class, 
    num_classes = num_classes,
    experiment_name='nn2_carlini_targeted_learningrate_0.05_initial_const_0.5_confidence_0.1_max_iter_1_to_7')

In [ ]:
sec_eval.plot()

In [ ]:
transfer_rates_percent = 100 * (1.0 - np.array(sec_eval._data['nb_correct']) / sec_eval._data['nb_pred'])

plt.plot(sec_eval.param_values, transfer_rates_percent, marker='o', color='red')
plt.xlabel(sec_eval.param_name)
plt.ylabel('Transferability Rate (%)')
plt.title('Transferability of Carlini&Wagner L2 Targeted attack to ResNet50')
plt.grid(True)
plt.show()